In [2]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
import json

In [ ]:
# Initialize an Azure OpenAI client
# 1. Load environment variables using load_dotenv() to access sensitive credentials securely.
# 2. Retrieve the API key, API version, and Azure endpoint from the environment variables.
# 3. Create an AzureOpenAI client instance using the retrieved credentials.

load_dotenv()
api_key= os.getenv("azure_open_api_key")
api_version= os.getenv("azure_open_api_version")
azure_endpoint= os.getenv("azure_open_api_endpoint")

client = AzureOpenAI(
  api_key = api_key,  
  api_version = api_version,
  azure_endpoint = azure_endpoint
)

In [3]:
# Create a generation prompt for user input samples
# 1. Define a detailed template for generating user inputs that trigger functions in a car control system.
# 2. Include specific instructions for:
#    - Structuring the user input with varying levels of explicitness about the function call.
#    - Incorporating emotional states to diversify the generated user inputs.
#    - Formatting outputs with unique identifiers and a strict output structure.
# 3. Use placeholders for dynamic values such as function name, description, parameters, explicitness, emotional state, and counter.
# 4. Ensure the prompt generates a wide variety of user inputs to account for different explicitness levels and emotional states.


generation_prompt = '''
- Generate user inputs that triggers a function for a car control system, in the form of a short sentence. Generate user prompts only in the context of car control. The objective function is defined by the following values:
-- function_name: {name},
-- function_description: {description},
-- function_parameters: {parameters}.
- Generate {nsamples} user inputs that refer explicitly to the parameter {selected_parameter}.
- Keep into account that some parameters are for the objective function may be required (must always appear in a function call) and others might not.
- We are interested in user inputs where the relationship between user inputs and objective function has different levels of explicitness.
-- An example of explicitness with value 1/5 (where the generated user_input must call the function in a quite implicit manner) for 'Adjust AC Temperature' would be: 'I hate winter weather'
-- An example of explicitness with value 5/5 (where the generated user_input must call the function in a quite explicit manner) for 'Adjust AC Temperature' would be: 'Set temperature to 24 degrees'
- The generated_user_input must have a explicitness relationship with the objective function of {explicitness}/{explicitness_steps}.
- Don't hesitate to occasionally use a casual language for the generated_user_input.
- Don't use in generated_user_input onomatopoeias such as 'Brrr!', 'Mmmm', or 'Ugh!'
- The emotional state influencing the generated_user_input must be be one of: {emotional_state}. The chosen emotional_state should be picked following an uniform distribution.
- Make sure to respect the explicitness relationship as well as the choice of emotional_state.
- We want to keep track of the ids of the generated samples, with a numeration that starts with counter={counter}. The id_number=counter+number_current_sample
- Make sure to generate_user_inputs that cover as many variations as possible in both what they say and in the parameters that they involve per objective function
- The format of a function call in chatGPT's function calling notation is always a dictionary with a first key "function name", with a second key "parameters".
- The strict format requirement for your outputs, without using code block markers or markdown formatting, is as follows per sample generated:
id_number|generated_user_input|function_name|{explicitness}/{explicitness_steps}|emotional_state|function_call with its parameters related to the generated_user_input in JSON format using chatGPT's function calling notation.
'''


In [53]:
# Define a function to generate a prompt using 

def get_prompts(user_prompt):
    """
    Generate a response based on the given user prompt using the Azure OpenAI GPT-4 model.

    Args:
        user_prompt (str): The prompt provided by the user to generate a response.

    Returns:
        str: The generated response from the GPT-4 model. Returns an empty string if an error occurs.
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "user", "content": user_prompt}, 
            ],
            temperature=0.2 
        )
        generated_prompts= response.choices[0].message.content
        return generated_prompts
    except:
        return ''

In [55]:
# Upload the function specification

syn_spec_path = './Data/function_spec'

# Read and load the JSON file in one step
with open(syn_spec_path, "r") as f:
    syn_spec = json.load(f)

# Convert the dictionary back to a JSON string
syn_spec_str = json.dumps(syn_spec)


In [ ]:
# Generate Synthetic Examples

# Define constants
explicitness_steps = 10
counter = 0
nsamples = 50
emotional_states = ['neutral', 'happy', 'angry', 'stressed', 'relaxed']

# Iterate over explicitness levels
for explicitness in range(10, 5, -1):
    for spec in syn_spec:
        # Extract function details
        name = spec['function']['name']
        description = spec['function']['description']
        parameters = spec['function']['parameters']['properties']
        selected_parameters = parameters.keys()

        # Iterate over parameters and emotional states
        for selected_parameter in selected_parameters:
            for emotional_state in emotional_states:
                # Generate the user prompt
                user_prompt = generation_prompt.format(
                    name=name,
                    description=description,
                    syn_spec_str=syn_spec_str,
                    selected_parameter=selected_parameter,
                    parameters=parameters,
                    explicitness=explicitness,
                    explicitness_steps=explicitness_steps,
                    emotional_state=emotional_state,
                    counter=counter,
                    nsamples=nsamples
                )
                # Generate prompts and write to file
                generated_prompts = get_prompts(user_prompt)
                filename = f"{name}_{selected_parameter}_{explicitness}_{emotional_state}.txt"
                output_path = f"./Data/tmp/{filename}"

                with open(output_path, "w") as f:
                    f.write(generated_prompts)

                print(f"Generated file: {filename}")
                counter += nsamples


In [57]:
# Initialize an empty DataFrame to accumulate all synthetic data
all_syn_data = pd.DataFrame()

# Define the path to the directory containing the generated examples
data_directory = './Data/tmp/'

# Iterate over all files in the specified directory
for filename in os.listdir(data_directory):
    file_path = os.path.join(data_directory, filename)
    
    # Open and read the content of each file
    with open(file_path, "r") as file:
        # Read all lines, remove empty lines, and process the data
        syn_data = [line.strip() for line in file.readlines() if line.strip()]
        
        # Split each line by '|' and convert it into a DataFrame
        syn_data_df = pd.DataFrame([line.split('|') for line in syn_data])
        
        # Concatenate the current DataFrame with the accumulated DataFrame
        all_syn_data = pd.concat([all_syn_data, syn_data_df], ignore_index=True)

# Assign meaningful column names to the final DataFrame
all_syn_data.columns = ['id', 'prompt', 'name', 'explicitness', 'emotional_state', 'ground_truth']

In [59]:
# Check for duplicate prompts and filter them out

# Group the data by 'prompt' and count the occurrences of each prompt
prompt_count = pd.DataFrame(all_syn_data.groupby(['prompt'])['id'].count()).reset_index()

# Filter out prompts that appear more than once, keeping only those with a single occurrence (unique prompts)
prompt_count = prompt_count[prompt_count['id'] == 1]

# Get the total number of unique prompts
unique_prompt_count = len(prompt_count)

# Filter the original data to retain only rows with unique prompts (those found in 'prompt_count')
all_syn_data_filtered = all_syn_data.merge(prompt_count[['prompt']], on='prompt', how='inner')

# Print the shape of the filtered DataFrame, showing the number of unique examples
print(f"Number of unique examples: {all_syn_data_filtered.shape[0]}")


Number of unique examples: 4898


In [60]:
# Group the filtered data by 'name' and count the number of prompts associated with each 'name'
prompt_count_by_name = all_syn_data_filtered.groupby('name')['prompt'].count()

# Display the result: number of prompts for each name
prompt_count_by_name

name
adjust_volume            998
change_radio_station     886
set_volume              1098
toggle_mute_radio        986
toggle_radio             929
Name: prompt, dtype: int64

In [49]:
# Get the unique function names from the 'name' column
functions = all_syn_data_filtered['name'].unique()

# Initialize empty DataFrames for storing training and testing data
train = pd.DataFrame()
test = pd.DataFrame()

# Iterate over each unique function name
for f in functions:
    # Filter the data for the current function
    fn_syn_data_filtered = all_syn_data_filtered[all_syn_data_filtered['name'] == f]
    
    # Shuffle the data for the current function using a fixed random seed for reproducibility
    fn_syn_data_filtered = fn_syn_data_filtered.sample(frac=1, random_state=89).reset_index(drop=True)
    
    # Split the shuffled data into training and testing sets (800 for training, 80 for testing)
    fn_train = fn_syn_data_filtered[:800]
    fn_test = fn_syn_data_filtered[800:880]
    
    # Concatenate the training and testing data to the main train and test DataFrames
    train = pd.concat((train, fn_train))
    test = pd.concat((test, fn_test))

In [61]:
# Print the shape of the training and testing DataFrames
print(f"Train DataFrame shape: {train.shape}")
print(f"Test DataFrame shape: {test.shape}")

Train DataFrame shape: (4000, 6)
Test DataFrame shape: (400, 6)


In [62]:
# Save the train DataFrame to a CSV file
train.to_csv('./Data/train_data.csv', index=False)

# Save the test DataFrame to a CSV file
test.to_csv('./Data/test_data.csv', index=False)

print("Train and test data saved successfully.")

Train and test data saved successfully.
